# Planning with EasyMCTS + `TCenterReward` — can the tree's edges reach *different* states?

**Problematic.** A Monte-Carlo tree search can only *search* if the edges leaving a
node lead to **distinguishable** states — if every sibling edge collapses back onto
the same world-model trajectory, UCB has nothing to choose between and the "plan"
degenerates to random. This notebook makes that concrete and quantitative:

1. **Part I — Where does edge diversity come from (and where does it go)?**
   We follow diversity through a *funnel* — action → latent state → task pose →
   reward — under the two levers the rollout primitives expose:
   *context-noise injection* for action diversity `p(a_t | o_t)` and for joint
   "imagine" diversity `p(o_{t+1}, a | o_t)`, plus the action-prior temperature.
   We measure not just *entropy* but the **effective number of distinguishable
   edges** (a Vendi score), how long an edge must be before it separates, a
   *recipe map* (which knobs actually produce branchable edges), and whether
   **reward diversity** and **task/decoded-state diversity** are redundant with
   latent diversity (they are **not** — that is the punchline).

2. **Part II — Plans with `EasyMCTS` + `TCenterReward` from a *non-optimal* start.**
   The reward (red **T** *centered & straight*) is deliberately simple — it is not
   the object of study; we use it to exercise planning with the world model. Goal:
   a plan that *raises* the reward from a bad starting configuration, and a sweep
   showing which planner parameters make planning beat random.

3. **Part III — Reading the search: tree metrics.** We compute the tree statistics
   that actually diagnose exploration (visit-count entropy, root-child value
   spread, realized branching/depth, the visit↔value correlation, the terminal-
   reward spread *inside* the tree) and show they mirror the Part-I edge-diversity
   diagnosis.

4. **Synthesis.** One picture tying it together: *planning beats random exactly
   when the root's edges are distinguishable.*

Everything runs in tokenizer-latent space under **bf16** (fp32 OOMs at planning
batch sizes) on the local GPU (`conda env dreamerv4uwm`).

## 0. Setup — model, data, helpers

In [ ]:
%load_ext autoreload
%autoreload 2
import math, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import mediapy
from torch.nn.functional import interpolate

torch.manual_seed(0)                       # reproducible noise priors
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
resolution = (256, 256)
print('device:', device)

In [ ]:
# --- config + checkpoints ---------------------------------------------------
# This notebook lives in dreamerv4uwm/planning/, so the diverse-tests relative
# `../scripts/config` no longer resolves. Anchor the hydra config dir to the
# installed package instead (robust to the kernel's working directory).
import dreamerv4uwm
from hydra import initialize_config_dir, compose

CONFIG_DIR = str(Path(dreamerv4uwm.__file__).resolve().parent.parent / 'scripts' / 'config')
with initialize_config_dir(version_base=None, config_dir=CONFIG_DIR):
    cfg = compose(config_name='dynamics/pushT-large',
                  overrides=['denoiser.horizon_aware=false'])

cfg.dynamics_ckpt  = '/home/mim-server/projects/rooholla/dreamerV4-UWM/checkpoints/blockcausal/pushT-post-train/97500.pt'
cfg.tokenizer_ckpt = '/home/mim-server/projects/rooholla/dreamerV4-UWM/checkpoints/tokenizer/pushT.pt'
print('config:', CONFIG_DIR)
print('horizon_aware:', cfg.denoiser.get('horizon_aware', False),
      '| n_actions:', cfg.denoiser.n_actions,
      '| latent grid:', cfg.denoiser.num_latent_tokens, 'x', cfg.denoiser.latent_dim)

In [ ]:
from dreamerv4uwm.models.utils import load_tokenizer, load_denoiser

denoiser  = load_denoiser(cfg, device, max_num_forward_steps=300).eval().cuda()
tokenizer = load_tokenizer(cfg, device, max_num_forward_steps=300).eval().cuda()
print('models loaded  |  frame_id_embedder:', denoiser.model.frame_id_embedder)

In [ ]:
# --- a held-out window -> latents (rich T motion; used for Part I) ----------
from dreamerv4uwm.datasets import ShardedHDF5Dataset

DATA_PATH = '/home/mim-server/datasets/pushT/h5/play'
dataset = ShardedHDF5Dataset(data_dir=DATA_PATH, window_size=64, stride=1,
                             split='train', train_fraction=0.9, split_seed=123)

batch   = dataset[1234]                                                       # fixed for reproducibility
imgs    = interpolate(batch['image'], resolution).to(device)[None]           # (1,T,3,256,256)
actions = batch['action'][:, :cfg.denoiser.n_actions][None].to(device)       # (1,T,n_act)
with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    latents = tokenizer.encode(imgs).float()                                 # (1,T,N_lat,D_lat)
print('imgs', tuple(imgs.shape), '| actions', tuple(actions.shape), '| latents', tuple(latents.shape))

In [ ]:
# --- display / decode / scoring helpers ------------------------------------
@torch.no_grad()
def decode(lat):
    """Latents (B,T,N,D) -> video (B,T,3,H,W) float[0,1] (bf16 autocast)."""
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        v = tokenizer.decode(lat.to(device))
    return v.float().clamp(0, 1)

@torch.no_grad()
def rgb_batch(z_states):
    """Terminal latents (B,N,D) -> (B,H,W,3) uint8 RGB (one batched decode)."""
    v = decode(z_states[:, None])[:, 0]                       # (B,3,H,W)
    return (v.permute(0, 2, 3, 1).clamp(0, 1) * 255).to(torch.uint8).cpu().numpy()

def frame_rgb(lat_1frame):
    """(1,1,N,D) latent -> (H,W,3) uint8 RGB."""
    v = decode(lat_1frame)[0, 0]
    return (v.permute(1, 2, 0).clamp(0, 1) * 255).to(torch.uint8).cpu().numpy()

def _strip(video_tchw, n_frames=8, border=2):
    T = video_tchw.shape[0]
    idx = np.linspace(0, T - 1, min(n_frames, T), dtype=int)
    fr = (video_tchw[idx].cpu().permute(0, 2, 3, 1).float().numpy() * 255).clip(0, 255).astype(np.uint8)
    H, W, C = fr.shape[1:]
    sep = np.full((H, border, C), 255, np.uint8)
    parts = []
    for i, f in enumerate(fr):
        if i: parts.append(sep)
        parts.append(f)
    return np.concatenate(parts, axis=1)

def plotComparison(named_videos, n_frames=8, title=None):
    """Stack labeled (T,C,H,W) clips as filmstrip rows."""
    rows = [(lab, _strip(v, n_frames)) for lab, v in named_videos]
    fig, axes = plt.subplots(len(rows), 1, figsize=(n_frames * 2, 2.0 * len(rows)))
    if len(rows) == 1: axes = [axes]
    for ax, (lab, img) in zip(axes, rows):
        ax.imshow(img); ax.set_xticks([]); ax.set_yticks([])
        ax.set_ylabel(lab, rotation=0, ha='right', va='center', fontsize=10)
    if title: axes[0].set_title(title, fontsize=12)
    plt.tight_layout(); plt.show()

print('helpers ready')

In [ ]:
# --- context / horizon split + a "make a decision state" helper ------------
T_ctx = 8                                     # conditioning frames a node holds

def make_state(t0, Tc=T_ctx):
    """(ctx_z, ctx_a, gt_next_action) for a decision at frame t0+Tc of the demo window."""
    cz = latents[:, t0:t0 + Tc].clone()
    ca = actions[:, t0:t0 + Tc].clone()
    gt = actions[:, t0 + Tc].clone()          # (1,n_act) dataset action
    return cz, ca, gt

# knobs — keep light for interactive use; dial up for final figures
FAST      = True
N_SAMPLES = 48 if FAST else 96               # edges / action samples per estimate
K_STEPS   = 8  if FAST else 12               # Euler steps per rollout
print(f'T_ctx={T_ctx}  N_SAMPLES={N_SAMPLES}  K_STEPS={K_STEPS}  FAST={FAST}')

## 1. A diversity-metrics library

The core question of the whole notebook — *do sibling edges reach different
states?* — is a **diversity** question, asked at four levels of a funnel:

| level | what it is | why it matters for MCTS |
|---|---|---|
| **action** `a` | the control the edge applies | the raw knob we can perturb |
| **latent state** `z` | the world-model state the child node *holds* | what the tree branches into |
| **task pose** | decoded T centroid `(x,y)` + upright-ness | the task-relevant part of `z` (latent diversity can be task-*irrelevant*) |
| **reward** `r` | the scalar the planner selects on | if it is flat, the tree cannot decide |

Beyond spread/entropy we compute the **effective number of distinguishable
samples** with a *Vendi score* — the exponentiated entropy of the eigenvalues of
an RBF similarity matrix (bandwidth = median pairwise distance). It reads as "how
many effectively-distinct edges are here": `1` = all collapsed, `B` = all distinct.
This is the single most on-target statistic for the problematic.

In [ ]:
# --- diversity primitives ---------------------------------------------------
def _to2d(X):
    if torch.is_tensor(X): X = X.detach().float().cpu().numpy()
    else:                  X = np.asarray(X, dtype=np.float64)
    return X.reshape(X.shape[0], -1).astype(np.float64)

def pairwise_dist(X):
    """(B,...) -> (B,B) euclidean distances over the flattened features."""
    Xt = torch.from_numpy(_to2d(X)).float()
    return torch.cdist(Xt, Xt).double().numpy()

def mean_pairwise_dist(X):
    D = pairwise_dist(X); B = D.shape[0]
    return float(D.sum() / (B * (B - 1) + 1e-9))

def vendi_score(X, sigma=None):
    """Effective number of distinguishable samples in [1, B] (RBF-kernel Vendi)."""
    Xf = _to2d(X); B = Xf.shape[0]
    if B < 2: return 1.0
    D = pairwise_dist(Xf); off = D[~np.eye(B, dtype=bool)]
    if sigma is None:
        pos = off[off > 0]; sigma = float(np.median(pos)) if pos.size else 1.0
    if sigma <= 0: return 1.0
    K = np.exp(-(D ** 2) / (2 * sigma ** 2))
    w = np.linalg.eigvalsh(K / B); w = w[w > 1e-12]
    return float(np.exp(-(w * np.log(w)).sum()))

def participation_ratio(X):
    """Effective dimensionality of the sample cloud (covariance eigenvalues)."""
    Xf = _to2d(X); Xc = Xf - Xf.mean(0, keepdims=True)
    lam = np.linalg.svd(Xc, compute_uv=False) ** 2
    return float(lam.sum() ** 2 / (lam ** 2).sum()) if lam.sum() > 0 else 0.0

def gaussian_entropy(samples):
    """Differential entropy (nats) of a Gaussian fit to samples (B,d) — small d only."""
    x = _to2d(samples); d = x.shape[1]
    cov = np.cov(x.T).reshape(d, d) + 1e-9 * np.eye(d)
    return float(0.5 * (d * math.log(2 * math.pi * math.e) + np.linalg.slogdet(cov)[1]))

# sanity: identical -> ~1, orthonormal -> ~B
_id = torch.zeros(6, 4); _orth = torch.eye(6)[:, :4] * 5
print(f'vendi(identical)={vendi_score(_id):.2f}  vendi(spread)={vendi_score(_orth):.2f}  (B=6)')

In [ ]:
from dreamerv4uwm.planning import (rollout as R, EasyMCTS, EasyPlanConfig,
                                   TCenterReward, score_t_centered, annotate_t)

# score kwargs shared by the reward and the task-pose readout (kept identical so
# the reward we report IS TCenterReward's value, computed from the same decode).
SCORE_KW = dict(center_xy=(0.5, 0.5), sigma=0.25, w_center=0.6, w_orient=0.4,
                orient_method='vertical')

def task_poses(rgb_np, **kw):
    """(B,H,W,3) uint8 -> (poses (m,3): [cx_norm, cy_norm, lr_sym], scores (B,))."""
    kw = {**SCORE_KW, **kw}
    scores, poses = [], []
    Himg, Wimg = rgb_np.shape[1:3]
    for b in range(rgb_np.shape[0]):
        s, d = score_t_centered(rgb_np[b], **kw)
        scores.append(s)
        if d.get('found'):
            cx, cy = d['centroid']
            poses.append([cx / Wimg, cy / Himg, d.get('lr_sym', 0.0)])
    return np.array(poses), np.array(scores)

def characterize_edges(cz, ca, *, H, B, ctx_noise=0.0, action_temp=1.0,
                       K=None, mode='imagine', seed=0, task=True):
    """Sample B edges from a node and measure diversity at every funnel level.
    Returns (metrics_dict, (z (B,H,N,D), a (B,H,n_act)))."""
    K = K or K_STEPS
    g = torch.Generator(device=device).manual_seed(seed)
    if mode == 'two_stage':                          # p(a|o) then world-model s,a->s'
        a = R.policy(denoiser, cz, ca, H, B=B, K=K, ctx_noise=ctx_noise,
                     action_temp=action_temp, generator=g)
        z = R.transition(denoiser, cz, ca, a, K=K, ctx_noise=ctx_noise, generator=g)
    else:                                            # joint imagine  p(o',a|o)
        z, a = R.imagine(denoiser, cz, ca, H, B=B, K=K, ctx_noise=ctx_noise,
                         action_temp=action_temp, generator=g)
    z_term = z[:, -1]                                # (B,N,D) child states
    m = dict(H=H, B=B, ctx_noise=ctx_noise, action_temp=action_temp, mode=mode)
    a0 = a[:, 0]                                     # first action of each edge
    m['act_mpd'] = mean_pairwise_dist(a0); m['act_vendi'] = vendi_score(a0)
    m['act_entropy'] = gaussian_entropy(a0)
    m['act_spread'] = float(a.std(0).norm(dim=-1).mean())
    m['lat_spread'] = float(z_term.std(0).mean()); m['lat_mpd'] = mean_pairwise_dist(z_term)
    m['lat_vendi'] = vendi_score(z_term); m['lat_pr'] = participation_ratio(z_term)
    if task:
        P, r = task_poses(rgb_batch(z_term))
        m['rew_mean'] = float(r.mean()); m['rew_std'] = float(r.std())
        m['rew_range'] = float(r.max() - r.min()); m['task_found'] = len(P)
        if len(P) >= 2:
            m['task_centroid_spread'] = float(np.linalg.norm(P[:, :2].std(0)))
            m['task_sym_std'] = float(P[:, 2].std()); m['task_vendi'] = vendi_score(P)
        else:
            m['task_centroid_spread'] = 0.0; m['task_sym_std'] = 0.0; m['task_vendi'] = 1.0
    return m, (z, a)

print('characterize_edges ready — measures action / latent / task / reward diversity per node')

# Part I — Where does edge diversity come from (and where does it go)?

The world model is the binding constraint: the policy `p(a|o)` is often a tight
blob, and even when we *widen* the action distribution the model tends to
**contract** those diverse actions back toward one short-horizon trajectory. So
"how do we generate edges that lead to different states?" is really: *which knobs
survive the funnel down to distinguishable states / rewards, and at what horizon?*

## 1.1 Action diversity `p(a_t | o_t)` — context-noise injection

`ctx_noise` mixes noise into the **observation** context. *Honest* injection also
tells the model the state is that uncertain (`sigma_idx` set to the true noised
cleanness) — a principled widening of the posterior; *mismatched* corrupts the
context but announces it clean (OOD contrast). We read out the action cloud, its
Gaussian entropy, and its **effective mode count** (Vendi).

In [ ]:
t0 = 24
cz, ca, gt = make_state(t0)
NOISES = [0.0, 0.15, 0.3, 0.5]

clouds, ent_h, ent_m, vend_h = {}, [], [], []
for n in NOISES:
    g = torch.Generator(device=device).manual_seed(0)
    a_h = R.policy(denoiser, cz, ca, H=1, B=N_SAMPLES, K=K_STEPS,
                   ctx_noise=n, ctx_noise_honest=True,  generator=g)[:, 0]
    g = torch.Generator(device=device).manual_seed(0)
    a_m = R.policy(denoiser, cz, ca, H=1, B=N_SAMPLES, K=K_STEPS,
                   ctx_noise=n, ctx_noise_honest=False, generator=g)[:, 0]
    clouds[f'n={n}'] = a_h.cpu().numpy()
    ent_h.append(gaussian_entropy(a_h)); ent_m.append(gaussian_entropy(a_m))
    vend_h.append(vendi_score(a_h))
    print(f'ctx_noise={n:.2f}  H_honest={ent_h[-1]:+.2f}  H_mismatch={ent_m[-1]:+.2f}  '
          f'vendi_honest={vend_h[-1]:.1f}/{N_SAMPLES}')

fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
for lab, c in clouds.items():
    ax[0].scatter(c[:, 0], c[:, 1], s=12, alpha=0.45, label=lab)
ax[0].scatter([gt[0, 0].item()], [gt[0, 1].item()], c='k', marker='*', s=220, label='dataset $a_t$', zorder=5)
ax[0].set_xlabel('action[0]'); ax[0].set_ylabel('action[1]'); ax[0].legend(fontsize=8)
ax[0].grid(alpha=.3); ax[0].set_title(f'honest ctx-noise action clouds (t0={t0})')
ax[1].plot(NOISES, ent_h, 'o-', label='honest'); ax[1].plot(NOISES, ent_m, 's--', label='mismatched')
ax[1].set_xlabel('ctx_noise'); ax[1].set_ylabel('action entropy (nats)'); ax[1].legend(); ax[1].grid(alpha=.3)
ax[1].set_title('entropy vs context noise')
ax[2].plot(NOISES, vend_h, 'o-', color='C2'); ax[2].set_xlabel('ctx_noise')
ax[2].set_ylabel('effective #distinct actions'); ax[2].grid(alpha=.3)
ax[2].set_title(f'Vendi score (max={N_SAMPLES})')
plt.tight_layout(); plt.show()

## 1.2 Action-prior temperature

In [ ]:
TEMPS = [0.5, 1.0, 1.5, 2.0, 2.5]
clouds_t, ent_t, vend_t = {}, [], []
for s in TEMPS:
    g = torch.Generator(device=device).manual_seed(0)
    a = R.policy(denoiser, cz, ca, H=1, B=N_SAMPLES, K=K_STEPS, action_temp=s, generator=g)[:, 0]
    clouds_t[f'T={s}'] = a.cpu().numpy()
    ent_t.append(gaussian_entropy(a)); vend_t.append(vendi_score(a))
    print(f'action_temp={s}  entropy={ent_t[-1]:+.2f}  vendi={vend_t[-1]:.1f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 4.3))
for lab, c in clouds_t.items():
    ax[0].scatter(c[:, 0], c[:, 1], s=12, alpha=0.45, label=lab)
ax[0].scatter([gt[0, 0].item()], [gt[0, 1].item()], c='k', marker='*', s=220, zorder=5)
ax[0].set_xlabel('action[0]'); ax[0].set_ylabel('action[1]'); ax[0].legend(fontsize=8)
ax[0].grid(alpha=.3); ax[0].set_title('action-prior temperature clouds')
ax[1].plot(TEMPS, ent_t, 'o-', label='entropy (nats)')
ax[1].plot(TEMPS, vend_t, 's-', label='vendi (#distinct)')
ax[1].set_xlabel('action_temp'); ax[1].legend(); ax[1].grid(alpha=.3)
ax[1].set_title('spread vs temperature  (temp!=1 is mildly OOD)')
plt.tight_layout(); plt.show()

## 1.3 The diversity funnel — action → latent → task → reward

Now the whole funnel at once, using joint `imagine` edges of a fixed horizon and
sweeping `ctx_noise`. Each curve is normalised to its `n=0` value so we can see
**which levels of diversity actually grow**. The expected story: actions widen a
lot, latent states less, and *task pose / reward* least — the world model
contracts the diversity that the planner needs.

In [ ]:
H_EDGE = 8
funnel = [characterize_edges(cz, ca, H=H_EDGE, B=N_SAMPLES, ctx_noise=n,
                             action_temp=1.5, seed=1)[0] for n in NOISES]
for m in funnel:
    print(f"ctx_noise={m['ctx_noise']:.2f}  act_vendi={m['act_vendi']:5.1f}  "
          f"lat_vendi={m['lat_vendi']:4.1f}  task_vendi={m['task_vendi']:4.1f}  "
          f"rew_std={m['rew_std']:.3f}  task_centroid_spread={m['task_centroid_spread']:.3f}")

def _norm(key):
    v = np.array([m[key] for m in funnel]); return v / (v[0] + 1e-9)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
for key, lab, mk in [('act_vendi', 'action', 'o-'), ('lat_vendi', 'latent state', 's-'),
                     ('task_vendi', 'task pose', '^-'), ('rew_std', 'reward std', 'd-')]:
    ax[0].plot(NOISES, _norm(key), mk, label=lab)
ax[0].set_xlabel('ctx_noise'); ax[0].set_ylabel('diversity relative to n=0')
ax[0].legend(); ax[0].grid(alpha=.3); ax[0].set_title(f'the diversity funnel (H={H_EDGE} edges)')
ax[1].plot(NOISES, [m['act_vendi'] for m in funnel], 'o-', label='action')
ax[1].plot(NOISES, [m['lat_vendi'] for m in funnel], 's-', label='latent state')
ax[1].plot(NOISES, [m['task_vendi'] for m in funnel], '^-', label='task pose')
ax[1].set_xlabel('ctx_noise'); ax[1].set_ylabel('effective # distinct edges (Vendi)')
ax[1].legend(); ax[1].grid(alpha=.3); ax[1].set_title('absolute distinguishable edges per level')
plt.tight_layout(); plt.show()

## 1.4 How long must an edge be before it separates?

Fix one batch of imagined rollouts and watch the effective number of distinct
states **grow with the frame index** `h`. If short edges collapse (Vendi ≈ 1) but
long edges fan out, then *edge length* is itself a diversity knob — directly
relevant to how we set `EasyPlanConfig.horizon`.

In [ ]:
B_DIV, H_DIV = (12 if FAST else 20), (16 if FAST else 24)
g = torch.Generator(device=device).manual_seed(2)
z_div, a_div = R.imagine(denoiser, cz, ca, H_DIV, B=B_DIV, K=K_STEPS,
                         ctx_noise=0.5, action_temp=1.5, generator=g)

state_vendi = [vendi_score(z_div[:, h]) for h in range(H_DIV)]
state_mpd   = [mean_pairwise_dist(z_div[:, h]) for h in range(H_DIV)]
vid = decode(z_div)                                        # (B,H,3,Hp,Wp), one decode
task_hs, task_vendi = list(range(0, H_DIV, 2)), []
for h in task_hs:
    rgb = (vid[:, h].permute(0, 2, 3, 1).clamp(0, 1) * 255).to(torch.uint8).cpu().numpy()
    P, _ = task_poses(rgb)
    task_vendi.append(vendi_score(P) if len(P) >= 2 else 1.0)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
ax[0].plot(range(H_DIV), state_vendi, 'o-', label='latent state')
ax[0].plot(task_hs, task_vendi, '^-', label='task pose')
ax[0].axhline(2, color='grey', ls=':', label='2 distinct (branch possible)')
ax[0].set_xlabel('edge frame h'); ax[0].set_ylabel('effective # distinct states')
ax[0].legend(); ax[0].grid(alpha=.3); ax[0].set_title('trajectories fan out with edge length')
ax[1].plot(range(H_DIV), state_mpd, 's-', color='C3')
ax[1].set_xlabel('edge frame h'); ax[1].set_ylabel('mean pairwise latent distance')
ax[1].grid(alpha=.3); ax[1].set_title('raw latent separation vs edge length')
plt.tight_layout(); plt.show()

plotComparison([(f'rollout {i}', decode(z_div[i:i+1])[0]) for i in range(min(4, B_DIV))],
               n_frames=H_DIV, title=f'do the {B_DIV} imagined edges actually differ? (ctx_noise=0.5, temp=1.5)')

## 1.5 A recipe map — which `(ctx_noise, horizon)` settings make branchable edges?

The deliverable of Part I: a heatmap of the **effective number of distinct
terminal states** and of the **terminal-reward spread** over the two knobs that
matter most for an MCTS edge. Bright = the tree can branch there; dark = edges
collapse and the search will be blind.

In [ ]:
GRID_NOISE = [0.0, 0.3, 0.5, 0.7]
GRID_H     = [2, 4, 8, 12]
lat_grid = np.zeros((len(GRID_NOISE), len(GRID_H)))
rew_grid = np.zeros_like(lat_grid)
Bg = 12 if FAST else 20
for i, n in enumerate(GRID_NOISE):
    for j, H in enumerate(GRID_H):
        m, _ = characterize_edges(cz, ca, H=H, B=Bg, ctx_noise=n, action_temp=1.5, seed=3)
        lat_grid[i, j] = m['lat_vendi']; rew_grid[i, j] = m['rew_std']
    print(f'ctx_noise={n:.2f} done')

fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
for a_, G, ttl in [(ax[0], lat_grid, f'effective # distinct terminal states (max {Bg})'),
                   (ax[1], rew_grid, 'terminal-reward std (what UCB sees)')]:
    im = a_.imshow(G, origin='lower', aspect='auto', cmap='viridis')
    a_.set_xticks(range(len(GRID_H)));    a_.set_xticklabels(GRID_H)
    a_.set_yticks(range(len(GRID_NOISE))); a_.set_yticklabels(GRID_NOISE)
    a_.set_xlabel('edge horizon H'); a_.set_ylabel('ctx_noise'); a_.set_title(ttl)
    for i in range(G.shape[0]):
        for j in range(G.shape[1]):
            a_.text(j, i, f'{G[i, j]:.2f}', ha='center', va='center',
                    color='w' if G[i, j] < G.max() * 0.6 else 'k', fontsize=9)
    fig.colorbar(im, ax=a_, fraction=0.046)
plt.suptitle('Edge-generation recipe map: where can the planner branch?', y=1.02)
plt.tight_layout(); plt.show()

## 1.6 Edge mode — joint `imagine` `p(o',a|o)` vs two-stage `p(a|o) → s,a→s'`

`EasyMCTS` can generate edges two ways. Joint imagination samples action and state
*together*; two-stage samples an action from the policy then rolls the world model
under it. Which gives more distinguishable children at equal cost?

In [ ]:
modes = ['imagine', 'two_stage']
mrows = {}
for md_ in modes:
    m, _ = characterize_edges(cz, ca, H=8, B=N_SAMPLES, ctx_noise=0.5, action_temp=1.5,
                              mode=md_, seed=4)
    mrows[md_] = m
    print(f"{md_:10s}  act_vendi={m['act_vendi']:5.1f}  lat_vendi={m['lat_vendi']:4.1f}  "
          f"task_vendi={m['task_vendi']:4.1f}  rew_std={m['rew_std']:.3f}")

labels = ['act_vendi', 'lat_vendi', 'task_vendi']
x = np.arange(len(labels)); w = 0.35
fig, ax = plt.subplots(figsize=(7.5, 4))
for k, md_ in enumerate(modes):
    ax.bar(x + (k - 0.5) * w, [mrows[md_][l] for l in labels], w, label=md_)
ax.set_xticks(x); ax.set_xticklabels(['action', 'latent', 'task pose'])
ax.set_ylabel('effective # distinct edges (Vendi)'); ax.legend()
ax.set_title('joint imagine vs two-stage edges — which branches more?')
plt.tight_layout(); plt.show()

## 1.7 Is reward diversity redundant with state diversity? (new idea)

The user's question: are *reward diversity* and *decoded-state diversity*
redundant with latent diversity? We test it directly. Across many decision states
and knob settings we scatter **latent Vendi** (x) against **terminal-reward std**
(y). If reward-std can be ≈ 0 while latent Vendi is large, then latent diversity
is partly *task-irrelevant* (the model moves the background / texture, not the T),
the **task-space and reward diversity are the binding ones**, and they are **not**
redundant. This is exactly the quantity UCB acts on, so it is the one to target.

In [ ]:
pts = []
STATES = [8, 16, 24, 32, 40]
for t0s in STATES:
    czs, cas, _ = make_state(t0s)
    for n in [0.3, 0.5, 0.7]:
        for H in [4, 8]:
            m, _ = characterize_edges(czs, cas, H=H, B=12 if FAST else 16,
                                      ctx_noise=n, action_temp=1.5, seed=5)
            pts.append(m)

lat_v = np.array([m['lat_vendi'] for m in pts])
task_v = np.array([m['task_vendi'] for m in pts])
rew_s = np.array([m['rew_std'] for m in pts])

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sc0 = ax[0].scatter(lat_v, rew_s, c=[m['H'] for m in pts], cmap='plasma', s=45)
ax[0].set_xlabel('latent Vendi (effective #distinct states)'); ax[0].set_ylabel('terminal-reward std')
ax[0].grid(alpha=.3); ax[0].set_title('latent diversity does NOT imply reward diversity')
fig.colorbar(sc0, ax=ax[0], label='edge horizon H')
sc1 = ax[1].scatter(task_v, rew_s, c=[m['ctx_noise'] for m in pts], cmap='viridis', s=45)
ax[1].set_xlabel('task-pose Vendi'); ax[1].set_ylabel('terminal-reward std')
ax[1].grid(alpha=.3); ax[1].set_title('task-pose diversity tracks reward diversity more tightly')
fig.colorbar(sc1, ax=ax[1], label='ctx_noise')
plt.tight_layout(); plt.show()

def _corr(a, b): return float(np.corrcoef(a, b)[0, 1])
print(f'corr(latent_vendi, reward_std) = {_corr(lat_v, rew_s):+.2f}')
print(f'corr(task_vendi,   reward_std) = {_corr(task_v, rew_s):+.2f}')
print('-> if the task correlation is higher, decoded/task diversity is the non-redundant signal.')

**Part I takeaways (fill in after running).** The action distribution is easy to
widen (honest `ctx_noise`, `action_temp`), but diversity *shrinks down the funnel*:
latent-state Vendi < action Vendi, and reward std smaller still. Edges only
separate once they are **long enough** (§1.4) and only in some `(ctx_noise, H)`
regions (§1.5). Reward/task diversity is **not** redundant with latent diversity
(§1.7) — it is the quantity the planner actually needs, so *that* is what edge
generation must target.

# Part II — Plans with `EasyMCTS` + `TCenterReward` from a non-optimal start

The reward is a simple pixel-space task score — the red **T** *centered & straight*
— used only to exercise planning; it is **not** the object of study. Goal: start
from a bad configuration and find a plan that **raises** the reward, then see which
planner parameters make planning beat random.

## 2.1 The reward and a genuinely non-optimal start

In [ ]:
reward_T = TCenterReward(decode_fn=decode, **SCORE_KW)

# A known non-optimal window (T shoved into the bottom-left corner). If it is not
# available, fall back to the lowest-reward frame in the demo window.
try:
    bad = dataset[24700]
    bimgs = interpolate(bad['image'][24:24 + T_ctx], resolution).to(device)[None]
    with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        bad_cz = tokenizer.encode(bimgs).float()
    bad_ca = bad['action'][24:24 + T_ctx, :cfg.denoiser.n_actions][None].to(device)
    src = 'dataset[24700] frames 24:32'
except Exception as e:
    print('fallback:', e)
    rs = [score_t_centered(frame_rgb(latents[:, t:t + 1]), **SCORE_KW)[0] for t in range(latents.shape[1])]
    t_bad = max(int(np.argmin(rs)), T_ctx)
    bad_cz = latents[:, t_bad - T_ctx:t_bad].clone(); bad_ca = actions[:, t_bad - T_ctx:t_bad].clone()
    src = f'demo frame {t_bad}'

s0, d0 = score_t_centered(frame_rgb(bad_cz[:, -1:]), **SCORE_KW)
plt.figure(figsize=(4, 4)); plt.imshow(annotate_t(frame_rgb(bad_cz[:, -1:]), d0)); plt.axis('off')
plt.title(f'non-optimal start ({src})\nR={s0:.2f}  center={d0["center"]:.2f}  straight={d0["orient"]:.2f}')
plt.tight_layout(); plt.show()
print(f'start reward = {s0:.3f}')

## 2.2 One plan, end to end

In [ ]:
epcfg = EasyPlanConfig(horizon=8, branching=5, sim_horizon=6, sim_rollouts=3,
                       n_iterations=20 if FAST else 32, max_depth=3, c_ucb=0.5,
                       gamma=0.98, n_min=0, K_steps=6, ctx_noise=0.6,
                       action_temp=2.0, max_ctx=24)
planner0 = EasyMCTS(denoiser, reward_T, epcfg, seed=0)
t = time.time(); out0 = planner0.plan(bad_cz, bad_ca, verbose=True)
print(f'planned in {time.time()-t:.1f}s | forwards={out0["n_forward"]} | nodes={len(planner0.all_nodes)}')

plan_z = torch.cat([e.z_seq for e in out0['best_path']], 0)[None]
plotComparison([('start ctx', decode(bad_cz)[0]), ('planned rollout', decode(plan_z)[0])],
               n_frames=8, title='EasyMCTS plan under the red-T reward')

def peak_along(z_states):
    """best TCenterReward over the frames of a rollout (MPC-style, matches Eq.5 max-over-prefix)."""
    rgb = rgb_batch(z_states); _, sc = task_poses(rgb); return float(sc.max()), sc

def random_peak(cfg_like, B=12, seed=7, extra=None):
    p = extra or {}
    g = torch.Generator(device=device).manual_seed(seed)
    zc = bad_cz.expand(B, -1, -1, -1).contiguous(); ac = bad_ca.expand(B, -1, -1).contiguous()
    peak = torch.full((B,), -1.0, device=device)
    for _ in range(p.get('max_depth', cfg_like.max_depth)):
        zz, aa = R.imagine(denoiser, zc, ac, p.get('horizon', cfg_like.horizon), B, K=6,
                           ctx_noise=p.get('ctx_noise', cfg_like.ctx_noise),
                           action_temp=p.get('action_temp', cfg_like.action_temp), generator=g)
        for h in range(zz.shape[1]):
            peak = torch.maximum(peak, reward_T(zz[:, h:h + 1]).squeeze(-1))
        zc = torch.cat([zc, zz], 1)[:, -24:]; ac = torch.cat([ac, aa], 1)[:, -24:]
    return peak.cpu().numpy()

plan_peak, _ = peak_along(plan_z[0])
rp = random_peak(epcfg)
print(f'planned peak reward = {plan_peak:.3f}   random peak = {rp.mean():.3f} ± {rp.std():.3f}   '
      f'(start {s0:.3f})  -> {"PLANNING WINS" if plan_peak > rp.mean()+rp.std() else "~ random"}')

## 2.3 Parameter sweep — which knobs make planning beat random?

For each config we compare the **planned** peak reward to a batch of **random**
`pi_prior` rollouts (same diversity knobs and depth), and we also record the
**root-edge reward std** from `characterize_edges` — the Part-I distinguishability
of the very first decision. We will use that in the synthesis.

In [ ]:
CONFIGS = {
 'short/tight':  dict(horizon=3,  branching=4, max_depth=2, sim_horizon=3,  sim_rollouts=2, ctx_noise=0.3, action_temp=1.0),
 'short/wide':   dict(horizon=3,  branching=5, max_depth=2, sim_horizon=3,  sim_rollouts=2, ctx_noise=0.6, action_temp=2.0),
 'long/tight':   dict(horizon=10, branching=5, max_depth=3, sim_horizon=8,  sim_rollouts=2, ctx_noise=0.3, action_temp=1.0),
 'long/wide':    dict(horizon=10, branching=6, max_depth=3, sim_horizon=8,  sim_rollouts=3, ctx_noise=0.6, action_temp=2.5),
}
sweep = {}
print(f'start reward = {s0:.3f}\n')
for name, p in CONFIGS.items():
    ep = EasyPlanConfig(K_steps=6, gamma=0.98, c_ucb=0.5, n_min=0, max_ctx=24,
                        n_iterations=16 if FAST else 28, **p)
    pl = EasyMCTS(denoiser, reward_T, ep, seed=0); o = pl.plan(bad_cz, bad_ca)
    pz = torch.cat([e.z_seq for e in o['best_path']], 0)[None]
    plan_peak, _ = peak_along(pz[0])
    rp = random_peak(ep, extra=p)
    # Part-I distinguishability of the root's edges under this config's edge knobs
    rm, _ = characterize_edges(bad_cz, bad_ca, H=p['horizon'], B=12,
                               ctx_noise=p['ctx_noise'], action_temp=p['action_temp'], seed=9)
    sweep[name] = dict(plan_peak=plan_peak, rand_mean=float(rp.mean()), rand_std=float(rp.std()),
                       gain=plan_peak - float(rp.mean()), root_rew_std=rm['rew_std'],
                       root_lat_vendi=rm['lat_vendi'], planner=pl, out=o, pz=pz)
    print(f"  {name:12s} plan_peak={plan_peak:.3f}  random={rp.mean():.3f}±{rp.std():.3f}  "
          f"gain={sweep[name]['gain']:+.3f}  root_rew_std={rm['rew_std']:.3f}")

names = list(sweep)
fig, ax = plt.subplots(figsize=(9, 4))
xx = np.arange(len(names))
ax.bar(xx - 0.2, [sweep[n]['plan_peak'] for n in names], 0.4, label='planned peak')
ax.bar(xx + 0.2, [sweep[n]['rand_mean'] for n in names], 0.4,
       yerr=[sweep[n]['rand_std'] for n in names], capsize=4, label='random peak')
ax.axhline(s0, color='k', ls='--', label=f'start reward ({s0:.2f})')
ax.set_xticks(xx); ax.set_xticklabels(names); ax.set_ylabel('peak TCenterReward'); ax.legend()
ax.set_title('does planning beat random? (per config)')
plt.tight_layout(); plt.show()

## 2.4 What the best plan actually does

In [ ]:
best_name = max(sweep, key=lambda n: sweep[n]['gain'])
pz = sweep[best_name]['pz']
full = torch.cat([bad_cz, pz], 1)
idxs = np.linspace(0, full.shape[1] - 1, 8, dtype=int)
strip = []
for t in idxs:
    rgb = frame_rgb(full[:, t:t + 1]); _, dbg = score_t_centered(rgb, **SCORE_KW)
    strip.append(annotate_t(rgb, dbg))
plt.figure(figsize=(16, 2.6)); plt.imshow(np.concatenate(strip, 1)); plt.axis('off')
plt.title(f'best plan ("{best_name}", gain={sweep[best_name]["gain"]:+.3f}) — ctx then planned rollout')
plt.tight_layout(); plt.show()

## 2.5 A working plan, visualized — non-optimal start → centered & vertical T

The corner start above is a *hard* case: the world model can barely move that T, so
planning ≈ random (§2.3). Planning **does** work where the model is controllable and
the edges are distinguishable — the thesis of Part I. An offline search over
`(start, config, seed)` (see `scratchpad/find_plan.py`) surfaced one such start;
here we reproduce it. From a T that starts on its side and off-center (`R≈0.13`),
`EasyMCTS` finds a plan that pushes it **upright and centered** (`R≈0.97`).

In [ ]:
# start chosen offline (low reward, high world-model reachability); recipe = "xlong/wide"
WIN, FR = 12000, 14
_b = dataset[WIN]
with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    win_cz = tokenizer.encode(interpolate(_b['image'][FR - T_ctx:FR], resolution).to(device)[None]).float()
win_ca = _b['action'][FR - T_ctx:FR, :cfg.denoiser.n_actions][None].to(device)

work_cfg = EasyPlanConfig(horizon=16, branching=5, max_depth=3, sim_horizon=12, sim_rollouts=3,
                          ctx_noise=0.7, action_temp=2.5, n_iterations=28, K_steps=6,
                          gamma=0.98, c_ucb=0.7, n_min=0, max_ctx=24)   # exact recipe from the search
t = time.time()
wout = EasyMCTS(denoiser, reward_T, work_cfg, seed=0).plan(win_cz, win_ca, verbose=True)
work_pz = torch.cat([e.z_seq for e in wout['best_path']], 0)[None]
wfull = torch.cat([win_cz, work_pz], 1)

wrgb, wrew = [], []
for tt in range(wfull.shape[1]):
    rgb = frame_rgb(wfull[:, tt:tt + 1]); s, d = score_t_centered(rgb, **SCORE_KW)
    wrgb.append(annotate_t(rgb, d)); wrew.append(s)
wrew = np.array(wrew); Tcw = win_cz.shape[1]; peak_t = int(Tcw + np.argmax(wrew[Tcw:]))
print(f'planned in {time.time()-t:.0f}s | start R={wrew[Tcw-1]:.3f} -> plan peak R={wrew[peak_t]:.3f} '
      f'(gain {wrew[peak_t]-wrew[Tcw-1]:+.3f})  plan length={work_pz.shape[1]} frames')

fig = plt.figure(figsize=(15, 6.4))
gs = fig.add_gridspec(2, 8, height_ratios=[1.25, 1.0], hspace=0.3, wspace=0.12)
samp = np.unique(np.concatenate([[Tcw - 1], np.linspace(Tcw, wfull.shape[1] - 1, 7, dtype=int)]))[:8]
for k, tt in enumerate(samp):
    ax = fig.add_subplot(gs[0, k]); ax.imshow(wrgb[tt]); ax.axis('off')
    tag = 'start' if tt == Tcw - 1 else ('PEAK' if tt == peak_t else f't={tt-Tcw+1}')
    ax.set_title(f'{tag}\nR={wrew[tt]:.2f}', fontsize=9,
                 color=('C2' if tt == peak_t else ('C3' if tt == Tcw - 1 else 'k')))
axc = fig.add_subplot(gs[1, 0:5])
axc.plot(range(wfull.shape[1]), wrew, 'o-', ms=4)
axc.axvspan(0, Tcw - 1, color='grey', alpha=0.15, label='context')
axc.axvline(Tcw - 0.5, color='grey', ls='--')
axc.scatter([Tcw - 1], [wrew[Tcw - 1]], c='C3', s=90, zorder=5, label=f'start R={wrew[Tcw-1]:.2f}')
axc.scatter([peak_t], [wrew[peak_t]], c='C2', s=120, marker='*', zorder=5, label=f'plan peak R={wrew[peak_t]:.2f}')
axc.set_xlabel('frame (context | planned rollout)'); axc.set_ylabel('TCenterReward')
axc.legend(fontsize=9, loc='lower right'); axc.grid(alpha=.3)
axc.set_title(f'MCTS plan raises the reward (gain {wrew[peak_t]-wrew[Tcw-1]:+.2f})')
axb = fig.add_subplot(gs[1, 5]); axb.imshow(wrgb[Tcw - 1]); axb.axis('off'); axb.set_title(f'START\nR={wrew[Tcw-1]:.2f}', color='C3')
axp = fig.add_subplot(gs[1, 6:8]); axp.imshow(wrgb[peak_t]); axp.axis('off'); axp.set_title(f'PLANNED (peak)\nR={wrew[peak_t]:.2f}', color='C2')
fig.suptitle('EasyMCTS + TCenterReward — a working plan (green=image center, yellow=T, magenta=T axis)', y=0.99)
plt.show()

# inline video of the planned rollout (holds a beat on the peak frame)
mediapy.show_video([np.ascontiguousarray(r) for r in wrgb] + [wrgb[peak_t]] * 6, fps=6)

# Part III — Reading the search: tree metrics

The tree statistics that actually diagnose exploration. The most on-target ones:

- **visit-count entropy** over the root's children (normalised to `[0,1]`): `1` =
  visits spread uniformly = the search could not tell the children apart; low =
  it committed. This is the *tree-level shadow* of edge collapse.
- **root-child value spread** (`max − mean`, and std): the decision resolution on
  the first action.
- **realized nodes per depth / mean visited depth / best-path depth**.
- **visit↔value correlation** across siblings: is UCB spending visits where the
  value is? (`nan` if too few visited children.)
- **edge-value (terminal reward) std inside the tree** vs a random batch: does the
  search preferentially expand distinguishable regions?

In [ ]:
def tree_metrics(planner, out=None):
    root = planner.root; ch = root.children
    visits = np.array([c.n_visit for c in ch], float)
    vals   = np.array([c.value if c.n_visit > 0 else np.nan for c in ch], float)
    tot = max(visits.sum(), 1.0); p = visits / tot; nz = p[p > 0]
    visit_entropy = float(-(nz * np.log(nz)).sum() / np.log(len(ch))) if len(ch) > 1 else 0.0
    fv = vals[np.isfinite(vals)]
    val_spread = float(fv.max() - fv.mean()) if fv.size else 0.0
    depths = [n.depth for n in planner.all_nodes]; maxd = max(depths)
    per_depth = {d: sum(x == d for x in depths) for d in range(maxd + 1)}
    visited = [n for n in planner.all_nodes if n.n_visit > 0]
    vc = [(c.n_visit, c.value) for c in ch if c.n_visit > 0]
    _vv = np.array([x[0] for x in vc], float); _qq = np.array([x[1] for x in vc], float)
    corr = (float(np.corrcoef(_vv, _qq)[0, 1])                 # is UCB spending visits where value is?
            if len(vc) > 2 and _vv.std() > 0 and _qq.std() > 0 else float('nan'))
    best_first = out['best_path'][0] if (out and out['best_path']) else max(ch, key=lambda c: c.value)
    ev = np.array([n.edge_val for n in planner.all_nodes if n.parent is not None])
    return dict(n_nodes=len(planner.all_nodes), n_root_children=len(ch),
                visit_entropy=visit_entropy, val_spread=val_spread, val_std=float(np.nanstd(vals)),
                per_depth=per_depth, mean_depth_visited=float(np.mean([n.depth for n in visited])) if visited else 0.0,
                best_depth=len(out['best_path']) if out else None,
                commit=float(best_first.n_visit / tot), visit_value_corr=corr,
                edge_val_std=float(ev.std()), edge_val_range=float(ev.max() - ev.min()),
                visits=visits, vals=vals)

tm0 = tree_metrics(planner0, out0)
for k, v in tm0.items():
    if k not in ('visits', 'vals'):
        print(f'  {k:20s} {v}')

## 3.1 Contrast: a *branchable* tree vs a *collapsed* tree

Run the search twice on the same start — once with edge knobs from Part I that
produce distinguishable edges (long/wide), once with knobs that collapse them
(short/tight) — and compare their tree metrics. The collapsed tree should show
**high visit entropy, ~zero value spread**: the planner is flying blind.

In [ ]:
CFG_BRANCH = dict(horizon=10, branching=6, max_depth=3, sim_horizon=8, sim_rollouts=3,
                  ctx_noise=0.6, action_temp=2.5)
CFG_COLLAPSE = dict(horizon=3, branching=6, max_depth=3, sim_horizon=3, sim_rollouts=2,
                    ctx_noise=0.0, action_temp=1.0)
trees = {}
for name, p in [('branchable', CFG_BRANCH), ('collapsed', CFG_COLLAPSE)]:
    ep = EasyPlanConfig(K_steps=6, gamma=0.98, c_ucb=0.5, n_min=0, max_ctx=24,
                        n_iterations=24 if FAST else 40, **p)
    pl = EasyMCTS(denoiser, reward_T, ep, seed=0); o = pl.plan(bad_cz, bad_ca)
    trees[name] = dict(m=tree_metrics(pl, o), pl=pl, o=o)
    m = trees[name]['m']
    print(f"{name:11s} visit_entropy={m['visit_entropy']:.2f}  val_spread={m['val_spread']:.3f}  "
          f"edge_val_std={m['edge_val_std']:.3f}  commit={m['commit']:.2f}  "
          f"visit~value corr={m['visit_value_corr']:.2f}")

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
# (a) root-child visits & values, branchable
for col, name in enumerate(['branchable', 'collapsed']):
    m = trees[name]['m']; ch = trees[name]['pl'].root.children
    order = np.argsort(-m['visits'])
    a = ax[0, col]
    a.bar(range(len(ch)), m['visits'][order], color='C0', alpha=0.6, label='visits')
    a.set_ylabel('root-child visits', color='C0'); a.set_xlabel('root child (sorted)')
    a.set_title(f"{name}:  visit_entropy={m['visit_entropy']:.2f}, val_spread={m['val_spread']:.3f}")
    a2 = a.twinx()
    a2.plot(range(len(ch)), m['vals'][order], 'o-', color='C3', label='value')
    a2.set_ylabel('root-child value', color='C3')
# (b) edge-value spread inside tree vs random batch
gb = torch.Generator(device=device).manual_seed(11)
zz, _ = R.imagine(denoiser, bad_cz, bad_ca, 8, 32, K=6, ctx_noise=0.6, action_temp=2.5, generator=gb)
rand_terminal = reward_T(zz[:, -1]).cpu().numpy()
for col, name in enumerate(['branchable', 'collapsed']):
    ev = np.array([n.edge_val for n in trees[name]['pl'].all_nodes if n.parent is not None])
    a = ax[1, col]
    a.hist(ev, bins=15, alpha=0.6, label='tree edge terminal reward', density=True)
    a.hist(rand_terminal, bins=15, alpha=0.4, label='random batch', density=True)
    a.set_xlabel('terminal reward'); a.set_ylabel('density'); a.legend(fontsize=8)
    a.set_title(f'{name}: edge_val_std={trees[name]["m"]["edge_val_std"]:.3f}')
plt.tight_layout(); plt.show()

**Reading the trees.** If the *collapsed* run has visit entropy near 1 and value
spread near 0 while the *branchable* run commits (lower entropy, positive value
spread, positive visit↔value correlation), then the tree metrics are just the
Part-I edge-diversity story observed through the search itself: **no
distinguishable edges → nothing to search.**

# Synthesis — planning helps exactly when the edges are distinguishable

One picture. For every Part-II config, the x-axis is the Part-I distinguishability
of the root's edges (terminal-reward std), the y-axis is the planning **gain over
random**. A positive trend is the thesis of the whole notebook.

In [ ]:
names = list(sweep)
xs = np.array([sweep[n]['root_rew_std'] for n in names])
ys = np.array([sweep[n]['gain'] for n in names])
cs = np.array([sweep[n]['root_lat_vendi'] for n in names])
plt.figure(figsize=(7.5, 5.5))
sc = plt.scatter(xs, ys, c=cs, cmap='viridis', s=140, edgecolor='k', zorder=3)
for n, x, y in zip(names, xs, ys):
    plt.annotate(n, (x, y), textcoords='offset points', xytext=(8, 4), fontsize=9)
plt.axhline(0, color='grey', ls='--')
if len(xs) > 2 and xs.std() > 0:
    b, a = np.polyfit(xs, ys, 1)
    xx = np.linspace(xs.min(), xs.max(), 20); plt.plot(xx, a + b * xx, 'C3-', alpha=.7,
        label=f'fit (slope={b:+.2f}, corr={np.corrcoef(xs, ys)[0,1]:+.2f})')
    plt.legend()
plt.colorbar(sc, label='root latent Vendi')
plt.xlabel('root-edge distinguishability  (terminal-reward std, Part I)')
plt.ylabel('planning gain over random  (Part II)')
plt.title('planning beats random exactly when the edges are distinguishable')
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## Findings, recipe & next steps

**Recipe for generating branchable edges** (fill numbers in after running):
- Widen the policy honestly: `ctx_noise ≈ 0.5–0.7` with `ctx_noise_honest=True`;
  `action_temp ≈ 1.5–2.5` adds spread but is mildly OOD.
- Make edges **long enough** — short edges collapse (§1.4); prefer the
  `(ctx_noise, horizon)` cells that light up in the recipe map (§1.5).
- Target **task/reward** diversity, not latent L2 — they are not redundant (§1.7);
  latent motion can be task-irrelevant.
- Pick start states / horizons where the world model is *controllable* — where the
  reward actually spreads across imagined rollouts.

**What still limits us.** The binding constraint is world-model controllability:
over short horizons the model contracts diverse actions back to one trajectory, so
the tree's edges collapse regardless of the search. The metrics here quantify
exactly *when* that happens.

**Next steps.**
1. KV-cached edge generation (`HybridChunkSampler`) to afford longer edges / deeper
   trees at the same cost.
2. A controllability probe: `d(task_pose)/d(action)` over the horizon, to pick
   states where planning has leverage — feed it back into edge generation.
3. Receding-horizon control: execute the best first chunk, advance the context,
   re-plan; measure closed-loop reward improvement from the non-optimal start.